# Pipeline + Reflection — конвейер с точечным усилением

Контентный конвейер, где этап написания усилен рефлексией. Исследователь и аналитик работают линейно — факты либо найдены, либо нет. А писатель работает в паре с критиком: цикл генератор-критик крутится, пока текст не будет одобрен или не исчерпан лимит итераций. Это принцип **точечного усиления** — рефлексия только там, где качество критично.

```mermaid
graph LR
    START((START)) --> researcher(["🔹 Researcher<br/><i>ищет</i>"])
    researcher --> analyst(["🔹 Analyst<br/><i>анализирует</i>"])
    analyst --> writing_team
    subgraph writing_team ["Writing Team — подграф рефлексии"]
        writer(["🔹 Writer<br/><i>пишет</i>"]) --> critic(["📊 Critic<br/><i>оценивает</i>"])
        critic -->|замечания| writer
    end
    writing_team -->|APPROVED| final_editor(["🔹 Final Editor<br/><i>пишет</i>"])
    final_editor --> END((END))

    classDef coordinator fill:#4A90D9,stroke:#2C5F8A,color:#fff,stroke-width:2px
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef aggregator fill:#F59E0B,stroke:#D97706,color:#fff,stroke-width:2px
    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff
    classDef subgraphStyle fill:#FDEBD0,stroke:#E67E22,stroke-width:2px,color:#333

    class START,END terminal
    class researcher,analyst,final_editor,writer worker
    class critic aggregator
```

In [1]:
import sys
sys.path.insert(0, "..")

from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Состояние конвейера

Состояние `ContentPipelineState` содержит семь полей. Линейные этапы (исследование, анализ) записывают результат в свои поля. Поля `draft` и `critique` обновляются в цикле рефлексии — каждая итерация перезаписывает их. Счётчик `writing_iteration` служит страховкой от бесконечного цикла.

In [3]:
llm = get_llm()

class ContentPipelineState(TypedDict):
    topic: str              # Тема контента
    research: str           # Результаты исследования
    analysis: str           # Аналитическая выжимка
    draft: str              # Черновик (обновляется в цикле)
    critique: str           # Критика текущего черновика
    final_content: str      # Финальный контент
    writing_iteration: int  # Счётчик итераций рефлексии

## Подграф рефлексии — writing_team

Цикл писатель-критик компилируется как **отдельный подграф**. Писатель создаёт или улучшает черновик; критик оценивает его и либо одобряет (ключевое слово `APPROVED`), либо возвращает конкретные замечания. Если текст не одобрен и лимит итераций не достигнут — управление возвращается к писателю.

Подграф использует тот же `ContentPipelineState`, что и основной конвейер, поэтому LangGraph бесшовно передаёт данные между уровнями.

In [4]:
def pipeline_writer_node(state: ContentPipelineState) -> dict:
    """Писатель: создаёт или улучшает черновик."""
    iteration = state.get("writing_iteration", 0) + 1
    critique = state.get("critique", "")

    if critique and iteration > 1:
        prompt = (
            f"Тема: {state['topic']}\nИсследование: {state['research']}\n"
            f"Анализ: {state['analysis']}\nТвой черновик: {state['draft']}\n"
            f"Замечания критика: {critique}\nИсправь текст."
        )
    else:
        prompt = (
            f"Тема: {state['topic']}\nИсследование: {state['research']}\n"
            f"Анализ: {state['analysis']}\nНапиши статью (5-6 предложений)."
        )

    response = llm.invoke([
        SystemMessage(content="Ты — профессиональный автор. Пиши качественные тексты на основе исследования и анализа."),
        HumanMessage(content=prompt)
    ])
    print(f"  [Писатель, итерация {iteration}] Текст готов")
    return {"draft": response.content, "writing_iteration": iteration}


def pipeline_critic_node(state: ContentPipelineState) -> dict:
    """Критик: оценивает черновик."""
    response = llm.invoke([
        SystemMessage(content="""Оцени текст строго. Проверь:
- Соответствие фактам из исследования
- Логичность аргументации
- Стиль и читаемость
Если текст хорош — ответь APPROVED. Иначе — конкретные замечания."""),
        HumanMessage(content=state["draft"])
    ])
    content = response.content
    if "APPROVED" in content.upper() or state.get("writing_iteration", 0) >= 3:
        print(f"  [Критик] ОДОБРЕНО")
        return {"critique": content, "final_content": state["draft"]}
    print(f"  [Критик] Замечания: {content[:60]}...")
    return {"critique": content}


def route_writing_critic(state: ContentPipelineState) -> str:
    if state.get("final_content"):
        return END
    return "writer"

### Компиляция подграфа

Подграф рефлексии компилируется отдельно. Условное ребро из критика ведёт либо обратно к писателю (замечания), либо к `END` (одобрено). Скомпилированный `writing_compiled` затем встраивается в основной конвейер как обычный узел.

In [5]:
writing_graph = StateGraph(ContentPipelineState)
writing_graph.add_node("writer", pipeline_writer_node)
writing_graph.add_node("critic", pipeline_critic_node)

writing_graph.add_edge(START, "writer")
writing_graph.add_edge("writer", "critic")
writing_graph.add_conditional_edges("critic", route_writing_critic, {
    "writer": "writer",
    END: END,
})

writing_compiled = writing_graph.compile()

## Основной конвейер — линейные узлы

Исследователь собирает факты, аналитик выделяет тренды и выводы, финальный редактор вычитывает готовый текст. Все три узла работают линейно — без циклов и рефлексии. Рефлексия применяется только на этапе написания, где качество текста критично.

In [6]:
def researcher_pipeline(state: ContentPipelineState) -> dict:
    """Исследователь: собирает ключевые факты по теме."""
    response = llm.invoke([
        SystemMessage(content="Найди 3-4 ключевых факта по теме. Кратко, по пунктам."),
        HumanMessage(content=state["topic"])
    ])
    print(f"  [1. Исследователь] Факты готовы")
    return {"research": response.content}


def analyst_pipeline(state: ContentPipelineState) -> dict:
    """Аналитик: анализирует факты, выделяет тренды и выводы."""
    response = llm.invoke([
        SystemMessage(content="Проанализируй факты: тренды, выводы. 3-4 тезиса."),
        HumanMessage(content=f"Тема: {state['topic']}\n\nФакты:\n{state['research']}")
    ])
    print(f"  [2. Аналитик] Анализ готов")
    return {"analysis": response.content}


def final_editor_node(state: ContentPipelineState) -> dict:
    """Финальный редактор: вычитка, форматирование, связность."""
    response = llm.invoke([
        SystemMessage(content="Финальная вычитка. Проверь форматирование, опечатки, общую связность."),
        HumanMessage(content=state["final_content"])
    ])
    print(f"  [4. Финальный редактор] Вычитка завершена")
    return {"final_content": response.content}

## Сборка конвейера

Основной конвейер линеен: `researcher` -> `analyst` -> `writing_team` -> `final_editor`. Но внутри узла `writing_team` скомпилированный подграф крутит цикл `writer` -> `critic` -> `writer`. LangGraph видит подграф как обычный узел — данные входят и выходят через общий `ContentPipelineState`.

In [7]:
pipeline = StateGraph(ContentPipelineState)
pipeline.add_node("researcher", researcher_pipeline)
pipeline.add_node("analyst", analyst_pipeline)
pipeline.add_node("writing_team", writing_compiled)       # Подграф рефлексии
pipeline.add_node("final_editor", final_editor_node)

pipeline.add_edge(START, "researcher")
pipeline.add_edge("researcher", "analyst")
pipeline.add_edge("analyst", "writing_team")
pipeline.add_edge("writing_team", "final_editor")
pipeline.add_edge("final_editor", END)

app = pipeline.compile()

## Запуск

Конвейер принимает тему и начинает работу: исследователь собирает факты, аналитик формулирует выводы, подграф рефлексии итеративно улучшает текст, финальный редактор вычитывает результат. Параметр `recursion_limit` ограничивает суммарное число шагов графа (включая внутренние итерации подграфа).

In [8]:
result = app.invoke(
    {"topic": "Будущее мультиагентных систем в enterprise", "writing_iteration": 0},
    config={"recursion_limit": 30},
)

print(f"\nИтераций рефлексии: {result['writing_iteration']}")
print(f"\nФинальный текст:\n{result['final_content'][:500]}...")

  [1. Исследователь] Факты готовы


  [2. Аналитик] Анализ готов


  [Писатель, итерация 1] Текст готов


  [Критик] ОДОБРЕНО


  [4. Финальный редактор] Вычитка завершена

Итераций рефлексии: 1

Финальный текст:
Текст в целом связный и хорошо структурирован. Я бы предложил лишь небольшие стилистические правки и унификацию формулировок.

Возможная отредактированная версия:

**Будущее мультиагентных систем в enterprise, скорее всего, будет строиться не вокруг полностью автономных ботов, а вокруг управляемой оркестрации — цепочек ролей с чёткими правилами, лимитами и аудитом. Наибольший практический эффект такие системы дадут в узких и повторяемых процессах: support, sales ops, procurement, ITSM и finance ...


## Итоги

**Pipeline + Reflection** реализует принцип точечного усиления: линейный конвейер дополнен циклом рефлексии только на критическом этапе (написание текста).

**Ключевые идеи:**
- Исследователь и аналитик работают линейно — факты либо найдены, либо нет, рефлексия здесь избыточна
- Писатель и критик образуют подграф с циклом — итеративное улучшение до одобрения или лимита
- Финальный редактор — линейный пост-процессинг для вычитки
- Подграф компилируется отдельно и встраивается в основной конвейер как обычный узел

**Когда использовать:** задачи с несколькими этапами, где один этап критичнее остальных и выигрывает от итеративного улучшения. Дешевле, чем Pipeline + Voting, но прицельно повышает качество там, где это важно.